In [ ]:
"""
Activity 1: Configuration Experiments for Deep Research System

This script runs multiple experiments with different configurations to compare
their performance and quality. Uses a hybrid evaluation approach:
- Automatic quantitative metrics (execution time, tokens, sources, etc.)
- LLM-as-a-judge comparative evaluation (single API call)

Experiments:
1. Increased Parallelism (max_concurrent_research_units: 10)
2. Deeper Research (max_researcher_iterations: 8, max_react_tool_calls: 15)
3. Anthropic Native Search (search_api: "anthropic")
4. Disabled Clarification (allow_clarification: False)
"""

============================================================================
## SECTION 1: IMPORTS AND SETUP
============================================================================

In [8]:
import os
import getpass
import asyncio
import time
import json
from pathlib import Path
from typing import Dict, Any, List
import uuid
# PDF processing
import PyPDF2
# LangChain and LangGraph imports
from langchain_anthropic import ChatAnthropic

# Import from open_deep_library
from open_deep_library.state import (
    AgentState,
    AgentInputState,
    SupervisorState,
    ResearcherState,
    ResearcherOutputState,
    ConductResearch,
    ResearchComplete,
    ClarifyWithUser,
    ResearchQuestion,
)

from open_deep_library.utils import (
    tavily_search,
    think_tool,
    get_all_tools,
    get_today_str,
)

from open_deep_library.configuration import (
    Configuration,
    SearchAPI,
)

from open_deep_library.prompts import (
    clarify_with_user_instructions,
    transform_messages_into_research_topic_prompt,
    lead_researcher_prompt,
    research_system_prompt,
    compress_research_system_prompt,
    final_report_generation_prompt,
)

from open_deep_library.deep_researcher import (
    clarify_with_user,
    write_research_brief,
    supervisor,
    supervisor_tools,
    researcher,
    researcher_tools,
    compress_research,
    final_report_generation,
    researcher_subgraph,
    supervisor_subgraph,
    deep_researcher,
)

In [ ]:
# Set up API keys
print("Setting up API keys...")
if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")
    
print("✓ API keys configured")

Setting up API keys...


============================================================================
## SECTION 2: HELPER FUNCTIONS
============================================================================

In [10]:
def load_pdf(pdf_path: str) -> str:
    """Load and extract text from PDF."""
    pdf_text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            pdf_text += page.extract_text() + "\n\n"
    return pdf_text

In [35]:
from IPython.display import Markdown, display


async def run_experiment(config: Dict[str, Any], experiment_name: str) -> Dict[str, Any]:
    """
    Run a single experiment and collect metrics.
    
    Args:
        config: Configuration dictionary for the experiment
        experiment_name: Name of the experiment for logging
        
    Returns:
        Dictionary containing:
        - final_report: The generated research report
        - metrics: Quantitative metrics (time, tokens, sources, etc.)
        - config: The configuration used
    """
    print(f"\n{'='*60}")
    print(f"Starting: {experiment_name}")
    print(f"{'='*60}")
    
    # Track metrics
    start_time = time.time()
    metrics = {
        "experiment_name": experiment_name,
        "execution_time": 0,
        "num_sources": 0,
        "report_length": 0,
        "supervisor_iterations": 0,
        "researchers_spawned": 0,
    }
    
        # Run the research and display output like the notebook
    final_state = None
    async for event in deep_researcher.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if node_output and "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")

            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
                
                # Track iterations
                if "research_iterations" in node_output:
                    metrics["supervisor_iterations"] = node_output["research_iterations"]
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
                    metrics["researchers_spawned"] = len(node_output["notes"])
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n{'='*60}")
                    print("FINAL REPORT GENERATED")
                    print(f"{'='*60}\n")
                    display(Markdown(node_output["final_report"]))
                    print(f"\n{'='*60}\n")
                    final_state = node_output

    
    # Calculate final metrics
    end_time = time.time()
    metrics["execution_time"] = round(end_time - start_time, 2)
    
    if final_state and "final_report" in final_state:
        report = final_state["final_report"]
        metrics["report_length"] = len(report)
        # Count sources (rough estimate - count lines starting with numbers or bullets)
        metrics["num_sources"] = report.count("\n- ") + report.count("\n1. ") + report.count("\n2. ")
        
        print(f"\n{'='*60}")
        print(f"✓ {experiment_name} COMPLETED")
        print(f"{'='*60}")
        print(f"Execution time: {metrics['execution_time']}s")
        print(f"Report length: {metrics['report_length']} characters")
        print(f"Sources found: {metrics['num_sources']}")
        print(f"Supervisor iterations: {metrics['supervisor_iterations']}")
        print(f"Researchers spawned: {metrics['researchers_spawned']}")
        print(f"{'='*60}\n")

        
        return {
            "final_report": report,
            "metrics": metrics,
            "config": config
        }
    else:
        print(f"\n✗ {experiment_name} failed to generate report")
        return {
            "final_report": "ERROR: No report generated",
            "metrics": metrics,
            "config": config
        }

In [12]:
def display_metrics_table(results: Dict[str, Dict[str, Any]]) -> None:
    """Display quantitative metrics in a formatted table."""
    print("\n" + "="*80)
    print("QUANTITATIVE METRICS COMPARISON")
    print("="*80)
    
    # Header
    print(f"\n{'Metric':<30} {'Exp 1':<12} {'Exp 2':<12} {'Exp 3':<12} {'Exp 4':<12}")
    print("-" * 80)
    
    # Execution time
    print(f"{'Execution Time (s)':<30} ", end="")
    for exp in ['exp1', 'exp2', 'exp3', 'exp4']:
        if exp in results:
            time_val = results[exp]['metrics']['execution_time']
            print(f"{time_val:<12.2f} ", end="")
    print()
    
    # Report length
    print(f"{'Report Length (chars)':<30} ", end="")
    for exp in ['exp1', 'exp2', 'exp3', 'exp4']:
        if exp in results:
            length = results[exp]['metrics']['report_length']
            print(f"{length:<12} ", end="")
    print()
    
    # Number of sources
    print(f"{'Number of Sources':<30} ", end="")
    for exp in ['exp1', 'exp2', 'exp3', 'exp4']:
        if exp in results:
            sources = results[exp]['metrics']['num_sources']
            print(f"{sources:<12} ", end="")
    print()
    
    # Supervisor iterations
    print(f"{'Supervisor Iterations':<30} ", end="")
    for exp in ['exp1', 'exp2', 'exp3', 'exp4']:
        if exp in results:
            iters = results[exp]['metrics']['supervisor_iterations']
            print(f"{iters:<12} ", end="")
    print()
    
    # Researchers spawned
    print(f"{'Researchers Spawned':<30} ", end="")
    for exp in ['exp1', 'exp2', 'exp3', 'exp4']:
        if exp in results:
            researchers = results[exp]['metrics']['researchers_spawned']
            print(f"{researchers:<12} ", end="")
    print()
    
    print("-" * 80)

In [13]:
async def evaluate_all_reports(results: Dict[str, Dict[str, Any]], baseline_report: str) -> str:
    """
    Use LLM to comparatively evaluate all reports.
    Single API call to rank all experiments.
    
    Args:
        results: Dictionary of experiment results
        baseline_report: The baseline report from the notebook
        
    Returns:
        String containing the comparative evaluation and rankings
    """
    print("\n" + "="*80)
    print("LLM COMPARATIVE EVALUATION")
    print("="*80)
    print("\nCalling Claude to evaluate and rank all reports...")
    
    # Prepare the evaluation prompt
    evaluation_prompt = f"""You are an expert research evaluator. You will compare 5 research reports and rank them from best to worst.

Evaluate based on these criteria:
1. Comprehensiveness - Does it cover all aspects of the research question?
2. Accuracy - Are findings well-supported and factually correct?
3. Clarity - Is it well-structured and easy to understand?
4. Depth - Does it provide meaningful insights beyond surface-level information?
5. Source Quality - Are sources credible and properly cited?

BASELINE REPORT (from original notebook):
{baseline_report[:3000]}...

EXPERIMENT 1 - Increased Parallelism (max_concurrent_research_units: 10):
{results['exp1']['final_report'][:3000]}...

EXPERIMENT 2 - Deeper Research (max_researcher_iterations: 8, max_react_tool_calls: 15):
{results['exp2']['final_report'][:3000]}...

EXPERIMENT 3 - Anthropic Native Search (search_api: "anthropic"):
{results['exp3']['final_report'][:3000]}...

EXPERIMENT 4 - Disabled Clarification (allow_clarification: False):
{results['exp4']['final_report'][:3000]}...

Please provide:
1. A ranking from 1st to 5th place
2. Brief justification for each ranking (2-3 sentences)
3. Overall winner and why
4. Key insights about which configuration works best for what scenarios

Format your response clearly with rankings and justifications."""

    # Call Claude for evaluation
    llm = ChatAnthropic(model="claude-sonnet-4-20250514", max_tokens=2000)
    response = await llm.ainvoke(evaluation_prompt)
    
    return response.content

In [14]:
def display_rankings(evaluation: str) -> None:
    """Display the LLM evaluation results."""
    print("\n" + "="*80)
    print("COMPARATIVE RANKINGS")
    print("="*80)
    print(f"\n{evaluation}")
    print("\n" + "="*80)

============================================================================
# SECTION 3: LOAD PDF AND CREATE RESEARCH QUESTION
============================================================================

In [15]:
print("\nLoading PDF document...")
pdf_path = "data/howpeopleuseai.pdf"
pdf_content = load_pdf(pdf_path)
print(f"✓ Loaded PDF with {len(pdf_content)} characters")


Loading PDF document...
✓ Loaded PDF with 112460 characters


In [17]:
# Create research request (same as notebook)
research_request = f"""
I have a PDF document about how people use AI. Please analyze this document and provide insights about:

1. What are the main findings about how people are using AI?
2. What are the most common use cases?
3. What trends or patterns emerge from the data?

Here's the PDF content:

{pdf_content[:10000]}  # First 10k chars to stay within limits

...[content truncated for context window]
"""

print("✓ Research question prepared")

✓ Research question prepared


============================================================================
## SECTION 4: BASELINE REFERENCE
============================================================================

NOTE: Baseline experiment already run in main notebook with config:
- max_concurrent_research_units: 1
- max_researcher_iterations: 2  
- max_react_tool_calls: 3
- search_api: "tavily"
- allow_clarification: True

The baseline report will be loaded from the notebook output for comparison.
All experiments below will be compared against those baseline results.

In [18]:
# For this script, we'll use a placeholder or load from file if available
baseline_report = """[PLACEHOLDER: Copy baseline report from notebook here, or load from file]

This should be the final report generated in the main notebook with the original configuration.
"""
print("\n" + "="*60)
print("BASELINE CONFIGURATION (from notebook)")
print("="*60)
print("- max_concurrent_research_units: 1")
print("- max_researcher_iterations: 2")
print("- max_react_tool_calls: 3")
print("- search_api: tavily")
print("- allow_clarification: True")
print("\nBaseline report will be used for comparison in LLM evaluation.")


BASELINE CONFIGURATION (from notebook)
- max_concurrent_research_units: 1
- max_researcher_iterations: 2
- max_react_tool_calls: 3
- search_api: tavily
- allow_clarification: True

Baseline report will be used for comparison in LLM evaluation.


In [24]:
results = {}


============================================================================
## SECTION 5: EXPERIMENT 1 - INCREASED PARALLELISM
============================================================================

In [25]:
print("\n" + "="*60)
print("EXPERIMENT 1: INCREASED PARALLELISM")
print("="*60)
print("Configuration: max_concurrent_research_units = 10")
print("Hypothesis: More parallel researchers = faster execution, broader coverage")


config_exp1 = {
    "configurable": {
        # Model configuration
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior - INCREASED PARALLELISM
        "allow_clarification": True,
        "max_concurrent_research_units": 10,  # ← CHANGED: 10 parallel researchers
        "max_researcher_iterations": 2,
        "max_react_tool_calls": 3,
        
        # Search configuration
        "search_api": "tavily",
        "max_content_length": 50000,
        
        # Thread ID
        "thread_id": str(uuid.uuid4())
    }
}

print("🔬 Running Experiment 1: Increased Parallelism...")
results['exp1'] = await run_experiment(config_exp1, "Experiment 1: Increased Parallelism")




EXPERIMENT 1: INCREASED PARALLELISM
Configuration: max_concurrent_research_units = 10
Hypothesis: More parallel researchers = faster execution, broader coverage
🔬 Running Experiment 1: Increased Parallelism...

Starting: Experiment 1: Increased Parallelism

Node: clarify_with_user

I have sufficient information to proceed with analyzing the NBER working paper "How People Use ChatGPT." I understand you want insights on: 1) Main findings about how people are using AI, 2) Most common use cases, and 3) Trends and patterns from the data. I have access to the PDF content which includes detailed research findings, data analysis, and classifications of ChatGPT usage patterns from May 2024 to June 2025. I will now begin analyzing this document to provide comprehensive insights on AI usage patterns, common use cases, and emerging trends.

Node: write_research_brief

Research Brief Generated:
I need a comprehensive analysis of the NBER working paper "How People Use ChatGPT" (Working Paper No. 34


Node: research_supervisor

Node: final_report_generation

FINAL REPORT GENERATED

# Comprehensive Analysis of NBER Working Paper: "How People Use ChatGPT"

## Main Findings About AI Usage Patterns

The [NBER working paper "How People Use ChatGPT"](https://www.nber.org/papers/w34255) [1] reveals unprecedented adoption rates and significant demographic shifts in AI usage. By July 2025, ChatGPT reached over 700 million weekly active users, representing approximately 10% of the global adult population, with 18 billion messages sent weekly [1][2]. This growth trajectory is historically unprecedented, reaching 1 million users within just 5 days of launch in November 2022 and hitting 100 million weekly active users within one year [4].

A critical finding is the dramatic shift from work-related to non-work usage. Non-work-related messages have grown from 53% in June 2024 to more than 70% by June 2025, while work-related usage dropped from 47% to 27% in the same period [1][2][5]. This shift r

============================================================================
## SECTION 6: EXPERIMENT 2 - DEEPER RESEARCH
============================================================================

In [32]:
print("\n" + "="*60)
print("EXPERIMENT 2: DEEPER RESEARCH")
print("="*60)
print("Configuration: max_researcher_iterations = 8, max_react_tool_calls = 15")
print("Hypothesis: More iterations = deeper insights, more comprehensive coverage")


config_exp2 = {
    "configurable": {
        # Model configuration
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior - DEEPER RESEARCH
        "allow_clarification": True,
        "max_concurrent_research_units": 1,
        "max_researcher_iterations": 8,  # ← CHANGED: More supervisor iterations
        "max_react_tool_calls": 15,      # ← CHANGED: More tool calls per researcher
        
        # Search configuration
        "search_api": "tavily",
        "max_content_length": 50000,
        
        # Thread ID
        "thread_id": str(uuid.uuid4())
    }
}

print("🔬 Running Experiment 2: Deeper Research...")
time.sleep(60) 
results['exp2'] = await run_experiment(config_exp2, "Experiment 2: Deeper Research")


EXPERIMENT 2: DEEPER RESEARCH
Configuration: max_researcher_iterations = 8, max_react_tool_calls = 15
Hypothesis: More iterations = deeper insights, more comprehensive coverage
🔬 Running Experiment 2: Deeper Research...

Starting: Experiment 2: Deeper Research

Node: clarify_with_user

I have sufficient information to proceed with analyzing the NBER working paper "How People Use ChatGPT." I understand you want insights about: 1) main findings about how people use AI, 2) most common use cases, and 3) emerging trends and patterns from the data. The document provides comprehensive data on ChatGPT usage from November 2022 through July 2025, including usage patterns, demographics, work vs. non-work applications, and conversation classifications. I will now begin analyzing this research to provide the requested insights.

Node: write_research_brief

Research Brief Generated:
I need a comprehensive analysis of the NBER working paper "How People Use ChatGPT" by Aaron Chatterji et al. (Working

# Comprehensive Analysis of "How People Use ChatGPT": NBER Working Paper Findings

This analysis examines the groundbreaking NBER working paper "How People Use ChatGPT" by Aaron Chatterji et al. (Working Paper No. 34255, September 2025), which provides the first large-scale empirical study of AI chatbot usage patterns based on privacy-preserving analysis of ChatGPT conversations from November 2022 through July 2025.

## Main Findings: AI Usage Patterns and Demographics

### Unprecedented Adoption Rates

ChatGPT achieved extraordinary adoption milestones that represent the fastest technology diffusion in history. By July 2025, ChatGPT had been adopted by approximately 10% of the world's adult population, representing around 700 million weekly active users who collectively send 18 billion messages each week [1][2]. The platform reached more than 2.6 billion daily messages, equivalent to over 30,000 messages per second as of June 2025 [3].

The speed of adoption broke all historical precedents for new technologies. ChatGPT reached 1 million users within just 5 days of its November 2022 launch, hit 100 million weekly active users in November 2023 (less than one year), and weekly active users have been doubling every 7-8 months since then [3]. Daily message volume increased 5.8 times in the last year alone, with the platform reaching 1 billion daily messages in December 2024 - a milestone that took Google 8 years to achieve with search [3].

### Dramatic Demographic Shifts

The research reveals significant changes in user demographics over the study period. Initially, early adopters were disproportionately male, with over 80% of users having typically male names in the early months [3][4]. However, the gender gap has narrowed dramatically over time, with users having typically feminine names increasing from 37% in January 2024 to 52% by July 2025, indicating that gender parity has been achieved [4][5].

Age distribution shows that nearly half of all adult messages come from users under 26, suggesting strong adoption among younger demographics [2]. Educational and occupational patterns reveal that work usage is more common among educated users in highly-paid professional occupations, indicating differential adoption rates based on socioeconomic factors [1][2].

### Global Geographic Patterns

Geographic adoption patterns show interesting economic development correlations. The study found higher growth rates in lower-income countries compared to wealthier nations. By May 2025, ChatGPT adoption growth rates in the lowest income countries were over 4 times those in the highest income countries [5]. Usage has expanded significantly in middle-income countries, with countries like Brazil, South Korea, and the United States showing similar usage rates despite vastly different GDP per capita levels of $10,000, $34,000, and $86,000 respectively [3].

This democratization of AI access suggests that there is now minimal difference in ChatGPT usage between countries at the 50th versus 90th percentile of GDP per capita, indicating rapid global diffusion across economic boundaries [3].

## Use Case Analysis: Work vs. Non-Work Applications

### The Shift to Personal Usage

One of the most significant findings is the dramatic shift from work-related to non-work-related usage over the study period. While both categories showed continuous growth, non-work messages expanded much faster, growing from 53% to more than 70% of all usage between June 2024 and June 2025 [1][2][6]. This trend represents a fundamental change in how AI chatbots are being integrated into daily life, with personal, casual, and entertainment-focused conversations now dominating the platform [7].

The research indicates that approximately 30% of consumer usage is work-related while approximately 70% is non-work-related, with both categories continuing to grow over time [5]. Importantly, this shift is primarily due to changing usage patterns within existing user cohorts rather than changes in the composition of new ChatGPT users, suggesting that individual users are finding more personal applications for the technology over time [1].

### Three Primary Conversation Categories

The study's automated classification system identified three dominant conversation categories that collectively account for nearly 80% of all ChatGPT interactions:

**Practical Guidance (29% of conversations)** encompasses tutoring and teaching, how-to advice across various topics, and creative ideation. This category represents the most common use case and reflects users seeking actionable advice and guidance for real-world problems and learning objectives [2][6].

**Seeking Information (24% of conversations)** includes searches for factual information about people, current events, products, and recipes. This usage pattern appears to serve as a close substitute for traditional web search engines, suggesting that users view ChatGPT as an alternative information discovery mechanism [1][2].

**Writing (24% of conversations)** involves the automated production of emails, documents, and other communications, as well as editing, critiquing, summarizing, and translating text provided by users. This category highlights the unique capability of chatbots to generate digital outputs compared to traditional search engines [1][2].

### Work-Related Usage Patterns

Within work-related conversations, writing emerges as the dominant task category, accounting for 40% of work-related messages on average in June 2025 [2]. Notably, about two-thirds of work-related writing tasks involve editing existing text rather than creating entirely new content, suggesting that ChatGPT serves more as a collaborative writing partner than a replacement for human creativity [2].

The research reveals that approximately 81% of work-related messages involve obtaining and interpreting information and making decisions or solving problems, emphasizing ChatGPT's role in decision support rather than task automation [2]. This finding has significant implications for understanding the economic value proposition of AI chatbots in professional contexts.

### User Intent Classification

The study classified user messages by intent using an "Asking/Doing/Expressing" taxonomy. Overall, 49% of messages are "Asking" (seeking information for decision-making), 40% are "Doing" (requesting task completion), and 11% are "Expressing" (personal expression) [2]. For work-specific messages, the pattern shifts with 56% being "Doing" tasks, reflecting the more task-oriented nature of professional usage [2].

Contrary to popular assumptions about AI usage, computer programming represents only 4.2% of messages, while companionship and social-emotional uses account for under 2.5% of messages, indicating that these applications remain relatively niche [2].

## Trends and Economic Implications

### Usage Evolution Patterns

All user cohorts showed similar usage evolution patterns - relatively stable usage through 2024, followed by substantial increases beginning in early 2025 [3]. This pattern suggests that ChatGPT became significantly more useful or user-friendly during this period, leading to deeper integration into people's daily routines and workflows [3].

The headline trend is that non-work use is growing much faster than work use, with personal applications driving the majority of platform growth [8]. This finding challenges assumptions about AI adoption being primarily driven by workplace productivity applications and suggests that consumer applications represent the larger market opportunity.

### Demographic Democratization

The research documents rapid closure of demographic gaps in ChatGPT usage. Gender disparities that were pronounced in early adoption phases have largely disappeared, with usage patterns now resembling the general adult population by mid-2025 [5]. This democratization trend extends beyond gender to include geographic and potentially socioeconomic dimensions, suggesting that AI tools are becoming more accessible across different population segments.

### Economic Impact and Value Creation

The study concludes that ChatGPT provides economic value primarily through decision support, which is especially important in knowledge-intensive jobs [1][2]. Rather than replacing human workers, the technology appears to augment human decision-making capabilities by providing rapid access to information, generating text options, and offering analytical clarity [6].

However, the research also highlights potential inequality concerns. Benefits appear to disproportionately favor users with higher education levels and better employment situations, raising ongoing questions about access and fairness [6]. Professionals in higher-skill roles are more likely to benefit from AI tools in their work, potentially widening productivity gaps between different worker categories [7].

### Broader Economic Implications

While most economic analysis of AI has focused on productivity impacts in paid work, this research suggests that the impact on non-work activities (home production) may be on a similar or even larger scale [1]. This finding aligns with other research estimating substantial consumer surplus from generative AI, with one study calculating at least $97 billion in consumer surplus in 2024 alone in the United States [1].

The research team concludes that ChatGPT has evolved from a novelty tool to a mainstream technology that is becoming integrated into daily routines as both a writing partner and decision-making aid [6]. For a new technology, this speed of global diffusion has no historical precedent, suggesting that AI chatbots represent a fundamentally different category of technological innovation [2].

The study's methodology employed strict privacy safeguards using a Data Clean Room approach, where researchers analyzed over 1 million conversations through automated classifiers without any human seeing raw user data or personally identifiable information [3][2]. This privacy-preserving approach enabled comprehensive analysis while protecting user confidentiality, setting a potential standard for future AI usage research.

### Sources

[1] How People Use ChatGPT: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=5487080
[2] How People Use ChatGPT | NBER: https://www.nber.org/papers/w34255
[3] How People Use ChatGPT - by David Deming - Forked Lightning: https://forklightning.substack.com/p/how-people-use-chatgpt
[4] [PDF] How People Use ChatGPT - National Bureau of Economic Research: https://www.nber.org/system/files/working_papers/w34255/w34255.pdf
[5] How people are using ChatGPT | OpenAI: https://openai.com/index/how-people-are-using-chatgpt/
[6] How People Are Really Using ChatGPT - Mike Jeffs: https://mikejeffs.com/blog/how-people-are-really-using-chatgpt/
[7] OpenAI releases research on ChatGPT usage worldwide - LinkedIn: https://www.linkedin.com/posts/aaron-ronnie-chatterji_this-morning-the-openai-economic-research-activity-7373377911476649986-_n_s
[8] How People Actually Use ChatGPT — What 1.5M Conversations Tell Us: https://medium.com/@adnanmasood/how-people-actually-use-chatgpt-what-1-5m-conversations-tell-us-about-the-next-decade-of-software-ea603212b458




✓ Experiment 2: Deeper Research COMPLETED
Execution time: 230.82s
Report length: 11245 characters
Sources found: 0
Supervisor iterations: 0
Researchers spawned: 0



============================================================================
## SECTION 7: EXPERIMENT 3 - ANTHROPIC NATIVE SEARCH
============================================================================

In [33]:
print("\n" + "="*60)
print("EXPERIMENT 3: ANTHROPIC NATIVE SEARCH")
print("="*60)
print("Configuration: search_api = 'anthropic'")
print("Hypothesis: Native search integration may provide better quality results")


config_exp3 = {
    "configurable": {
        # Model configuration
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,
        "max_researcher_iterations": 2,
        "max_react_tool_calls": 3,
        
        # Search configuration - ANTHROPIC NATIVE SEARCH
        "search_api": "anthropic",  # ← CHANGED: Use Anthropic's native search
        "max_content_length": 50000,
        
        # Thread ID
        "thread_id": str(uuid.uuid4())
    }
}

print("🔬 Running Experiment 3: Anthropic Native Search...")
time.sleep(60)
results['exp3'] = await run_experiment(config_exp3, "Experiment 3: Anthropic Native Search")


EXPERIMENT 3: ANTHROPIC NATIVE SEARCH
Configuration: search_api = 'anthropic'
Hypothesis: Native search integration may provide better quality results
🔬 Running Experiment 3: Anthropic Native Search...

Starting: Experiment 3: Anthropic Native Search

Node: clarify_with_user

I have sufficient information to proceed with analyzing the ChatGPT usage report. Based on your request, I understand you want insights about:

1. Main findings about how people are using AI (specifically ChatGPT)
2. Most common use cases 
3. Trends and patterns from the data

The PDF content you've provided is from an NBER working paper titled "How People Use ChatGPT" that analyzes ChatGPT usage patterns from launch through July 2025. I can see key data points including growth statistics, work vs. non-work usage breakdown, and classification of conversation topics.

I will now begin analyzing this document to provide comprehensive insights on ChatGPT usage patterns, trends, and implications based on the research

RateLimitError: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': 'This request would exceed the rate limit for your organization (d8cedd58-98a3-4d1f-88ef-478f849dfcae) of 30,000 input tokens per minute. For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You may also contact sales at https://www.anthropic.com/contact-sales to discuss your options for a rate limit increase.'}, 'request_id': 'req_011CU7vKbTaQyKL98LKKAP6X'}

============================================================================
## SECTION 8: EXPERIMENT 4 - DISABLED CLARIFICATION
============================================================================

In [36]:
print("\n" + "="*60)
print("EXPERIMENT 4: DISABLED CLARIFICATION")
print("="*60)
print("Configuration: allow_clarification = False")
print("Hypothesis: Skipping clarification may speed up workflow but reduce accuracy")


config_exp4 = {
    "configurable": {
        # Model configuration
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior - DISABLED CLARIFICATION
        "allow_clarification": False,  # ← CHANGED: Skip clarification phase
        "max_concurrent_research_units": 1,
        "max_researcher_iterations": 2,
        "max_react_tool_calls": 3,
        
        # Search configuration
        "search_api": "tavily",
        "max_content_length": 50000,
        
        # Thread ID
        "thread_id": str(uuid.uuid4())
    }
}

print("🔬 Running Experiment 4: Disabled Clarification...")
time.sleep(60)
results['exp4'] = await run_experiment(config_exp4, "Experiment 4: Disabled Clarification")


EXPERIMENT 4: DISABLED CLARIFICATION
Configuration: allow_clarification = False
Hypothesis: Skipping clarification may speed up workflow but reduce accuracy
🔬 Running Experiment 4: Disabled Clarification...

Starting: Experiment 4: Disabled Clarification

Node: clarify_with_user

Node: write_research_brief

Research Brief Generated:
I need you to analyze the NBER Working Paper "How People Use ChatGPT" by Chatterji et al. (2025) and provide comprehensive insights about: (1) What are the main findings about how people are using AI, specifically focusing on the key patterns and behaviors identified in ChatGPT usage data from November 2022 through July 2025; (2) What are the most common use cases, including the breakdown of the three dominant categories (Practical Guidance, Seeking Information, and Writing) that account for nea...



Node: research_supervisor

Node: final_report_generation

FINAL REPORT GENERATED



# How People Use ChatGPT: Comprehensive Analysis of the NBER Working Paper

The NBER Working Paper "How People Use ChatGPT" by Chatterji et al. (2025) represents the first comprehensive, large-scale empirical study of how people actually use AI chatbots. This groundbreaking research analyzes approximately 1.5 million de-identified ChatGPT conversations from 2024-2025, revealing unprecedented insights into the adoption patterns, usage behaviors, and economic implications of AI technology that has achieved the fastest global diffusion in history.

## Unprecedented Scale and Growth Patterns

ChatGPT's adoption trajectory represents an extraordinary technological phenomenon. By July 2025, the platform had reached over 700 million weekly active users, representing approximately 10% of the world's adult population [1][2][3]. Users were sending 18 billion messages weekly, translating to more than 2.6 billion messages per day or over 30,000 messages per second [3][6]. This scale represents the fastest technology diffusion in history, with ChatGPT reaching 1 billion daily messages in December 2024—less than two years after its release—compared to Google Search, which took eight years to reach 1 billion daily searches [3][8].

The growth patterns reveal sustained acceleration rather than plateauing adoption. Total message volume increased by 5.8 times in just the last year of the study period, while user growth increased by 3.2 times, indicating that existing users are becoming more engaged over time [8]. Early adopter cohorts from Q1 2023 were sending 40% more messages per day than they did two years prior, while users who joined in late 2024 were sending nearly twice as many messages as their initial usage patterns suggested [8].

## The Shift from Work to Personal Use

One of the most significant findings concerns the dramatic shift in usage patterns between work-related and non-work applications. While both categories have grown continuously, non-work messages have experienced much faster growth, expanding from 53% of all usage in June 2024 to more than 70% by June 2025 [1][2][4][6][10]. This trend represents a fundamental shift in how people integrate AI into their daily lives, moving beyond purely professional applications to encompass a broad range of personal activities.

This shift toward non-work usage occurs primarily through changing behavior within existing user cohorts rather than compositional changes in new users. The research suggests this pattern reflects improvements in ChatGPT's capabilities and user-friendliness, making it more accessible and valuable for everyday personal tasks [8]. This finding challenges much of the economic analysis of AI that has focused primarily on workplace productivity, suggesting that the impact on non-work activities (home production) may be on a similar or even larger scale [6].

## The Three Dominant Use Cases

The research reveals that nearly 80% of all ChatGPT usage falls into three primary categories: Practical Guidance (29%), Seeking Information (24%), and Writing (24%) [1][2][4][6][10]. This concentration demonstrates remarkable consistency in how people choose to leverage AI capabilities across diverse contexts and demographics.

**Practical Guidance** emerges as the most common use case at 28.3% of all messages, encompassing tutoring and teaching activities, how-to advice across various topics, and creative ideation [2][7]. Notably, tutoring alone accounts for approximately 10% of all messages, highlighting ChatGPT's significant role in educational support [2]. This category reflects users' desire for personalized, actionable advice tailored to their specific circumstances and needs.

**Seeking Information** accounts for 24% of usage and includes searching for factual information, current events, products, and recipes [1][6]. The researchers note that this usage pattern "appears to be a very close substitute for web search," indicating that ChatGPT is directly competing with traditional search engines for information retrieval tasks [4]. The growth of this category signals a fundamental shift in how people access and process information, moving from keyword-based searches to conversational queries.

**Writing** represents 24% of overall usage and includes email drafting, document creation, editing, translation, and summarization [1][6]. Importantly, about two-thirds of all Writing messages ask ChatGPT to modify existing user text rather than creating entirely new content from scratch [6]. This pattern emphasizes ChatGPT's role as a collaborative tool that enhances human-generated content rather than simply replacing human creativity and expression.

## Work Usage Patterns and Professional Applications

While non-work usage dominates overall statistics, work-related applications reveal distinct patterns that illuminate ChatGPT's role in professional contexts. Writing dominates work-related tasks, accounting for 40% of work-related messages overall and highlighting chatbots' unique ability to generate digital outputs compared to traditional search engines [4][10][6][7].

The research employs a novel taxonomy classifying user intent as "Asking" (49% of messages), "Doing" (40% of messages), and "Expressing" (11% of messages) [6][7]. In work contexts, the distribution shifts significantly, with "Doing" activities comprising 56% of work-related messages, primarily consisting of writing tasks like editing, summarizing, and translating [1][7]. This pattern suggests that professional users leverage ChatGPT more heavily for task completion and output generation rather than advisory functions.

Work usage correlates strongly with education and occupation levels. Users with higher education levels and those in highly-paid professional occupations demonstrate significantly higher rates of work-related ChatGPT usage [2][4]. Approximately 58% of work-related messages associate with two broad workplace activities: obtaining, documenting, and interpreting information; and making decisions, giving advice, solving problems, and thinking creatively [2].

## Demographic Adoption Patterns and Digital Divide Narrowing

The demographic evolution of ChatGPT usage reveals remarkable patterns of digital divide narrowing across multiple dimensions. The most striking change concerns gender adoption patterns. Early adopters were overwhelmingly male, with around 80% of active users having masculine names in the first months after launch [2][4][6][8][10]. However, this gender gap has narrowed dramatically, with 52% of active users having typically female names by July 2025, indicating near complete gender parity and actually mirroring the broader adult population [1][3].

Age patterns show that young adults dominate usage, with people between ages 18 and 25 accounting for nearly half of all adult messages [2][6]. However, the age gaps have narrowed somewhat in recent months, suggesting broader age-based adoption [2]. Older users demonstrate different usage patterns, sending proportionally more work-related queries than their younger counterparts [2].

Perhaps most significantly from a global development perspective, usage has grown relatively faster in low- and middle-income countries compared to wealthier nations [2][4][6][8][10]. Growth rates in the lowest income countries are over 4 times higher than in the highest income countries as of May 2025 [2][3]. Countries like Brazil, South Korea, and the United States now demonstrate similar ChatGPT usage rates despite vastly different GDP per capita levels ($10,000, $34,000, and $86,000 respectively) [8]. This pattern suggests that AI technology may help reduce rather than exacerbate global digital divides.

## User Satisfaction and Engagement Quality

User satisfaction metrics indicate strong and improving engagement quality. Positive user ratings ("good" interactions) are now over four times more common than negative ones, demonstrating high user satisfaction across the platform [2][1]. "Asking" messages consistently receive higher quality ratings than other categories and have shown faster growth over the study period [4][7]. This pattern suggests that users increasingly value ChatGPT's advisory capabilities over pure task automation.

The growth in "Asking" relative to "Doing" activities across 2024-2025, combined with higher quality scores for advisory interactions, indicates that users are discovering the platform's strengths in providing personalized guidance and decision support [7]. This trend aligns with the researchers' conclusion that ChatGPT's primary economic value lies in its role as a decision-support tool rather than simply as a task automation system.

## Surprising Findings About Coding and Self-Expression

Contrary to much public discourse about AI's impact on programming and creative expression, the research reveals that these applications represent relatively small portions of actual usage. Computer programming accounts for only 4.2% of all messages, significantly lower than other studies and much smaller than public attention might suggest [1][6][3]. Similarly, self-expression and creative writing remain niche applications [3][10].

Relationships and personal reflection topics account for just 1.9% of messages, indicating that social or companionship uses remain quite limited despite significant media attention to these potential applications [4][8]. These findings suggest that actual usage patterns differ substantially from both promotional materials and concerns about AI replacing human creativity or social connection.

## Privacy-Preserving Methodology and Research Innovation

The study employed groundbreaking methodology that enables comprehensive analysis while maintaining strict privacy protections. No member of the research team ever saw actual message content or any personally identifiable information [3][6]. All analyses were conducted using automated LLM-based classifiers on de-identified, PII-scrubbed data, with employment and education data analyzed through secure data clean rooms with strict aggregation thresholds [6][7].

This privacy-preserving approach allowed researchers to analyze approximately 1.5 million de-identified conversations while ensuring user privacy [1][2]. The methodology employed automated classification taxonomies defined through prompts passed to LLMs, enabling systematic categorization of usage patterns without human review of message content [7]. The study received approval from Harvard IRB and represents a new model for conducting large-scale behavioral research on sensitive digital platforms [4].

## Economic Value and Broader Implications

The research documents substantial economic value creation through ChatGPT usage. The authors cite evidence that U.S. users would need compensation of roughly $98 to give up generative AI for a month, implying at least $97 billion in annual consumer surplus in 2024 alone [7]. This economic value derives primarily from ChatGPT's function as a decision-support tool, which proves especially important in knowledge-intensive jobs [1][2][4][6][10].

The shift toward advisory interactions rather than pure automation highlights the centrality of writing and communication in modern work processes [2]. Approximately 30% of consumer usage is work-related and 70% is non-work-related, with both categories continuing to grow over time, suggesting broad-based value creation across multiple life domains [7].

The researchers conclude that ChatGPT is transitioning from a technology novelty to core infrastructure that shapes how people think, work, and live, similar to how Google became ubiquitous in the mid-2000s [3]. The global scale of adoption, combined with sustained growth patterns and high user satisfaction, suggests that AI chatbots represent a fundamental shift in human-computer interaction rather than a temporary technological trend.

The study's findings have significant implications for productivity measurement, product design, and policy development. The documentation of globally broadening adoption with narrowing demographic divides suggests potential for AI technology to democratize access to sophisticated analytical and creative capabilities. However, the concentration of work usage among highly educated professionals in well-paid occupations also highlights potential concerns about amplifying existing economic advantages.

### Sources

[1] How People Really Use ChatGPT: Findings from NBER Research: https://techmaniacs.com/2025/09/15/how-people-really-use-chatgpt-findings-from-nber-research/
[2] How People Use ChatGPT - SSRN: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=5487080
[3] How People Use ChatGPT - by David Deming - Forked Lightning: https://forklightning.substack.com/p/how-people-use-chatgpt
[4] How People Use ChatGPT | NBER: https://www.nber.org/papers/w34255
[5] How People Actually Use ChatGPT — What 1.5M Conversations Tell Us: https://medium.com/@adnanmasood/how-people-actually-use-chatgpt-what-1-5m-conversations-tell-us-about-the-next-decade-of-software-ea603212b458
[6] How People Use ChatGPT - National Bureau of Economic Research: https://www.nber.org/system/files/working_papers/w34255/w34255.pdf
[7] How people are using ChatGPT | OpenAI: https://openai.com/index/how-people-are-using-chatgpt/
[8] OpenAI Study Uncovers 3 Surprises About How People Use ChatGPT: https://observer.com/2025/09/chatgpt-usage-openai-study/
[9] 1.5 million chats reveal who uses ChatGPT and why: https://searchengineland.com/chatgpt-who-why-openai-study-462008
[10] ChatGPT Study: 1 In 4 Conversations Now Seek Information: https://www.searchenginejournal.com/chatgpt-study-1-in-4-conversations-now-seek-information/556104/




✓ Experiment 4: Disabled Clarification COMPLETED
Execution time: 307.64s
Report length: 13734 characters
Sources found: 0
Supervisor iterations: 0
Researchers spawned: 0



In [ ]:
# Display metrics
display_metrics_table(results)

# Run LLM evaluation
evaluation = await evaluate_all_reports(results, baseline_report)
display_rankings(evaluation)

============================================================================
SECTION 9: MAIN EXECUTION
============================================================================

In [ ]:
async def main():
    """
    Main execution function that:
    1. Runs each experiment once
    2. Collects results as they complete
    3. Displays metrics summary
    4. Runs LLM evaluation once
    5. Shows final rankings
    """
    
    print("\n" + "="*80)
    print("STARTING ACTIVITY 1 EXPERIMENTS")
    print("="*80)
    print("\nThis will run 4 experiments with different configurations.")
    print("Each experiment will be tracked for metrics and evaluated by LLM.")
    print("\nEstimated time: 5-10 minutes depending on configurations")
    print("="*80)
    
    # Dictionary to store all results
    results = {}
    
    # Run Experiment 1
    print("\n🔬 Running Experiment 1: Increased Parallelism...")
    results['exp1'] = await run_experiment(config_exp1, "Experiment 1: Increased Parallelism")
    
    # Run Experiment 2
    print("\n🔬 Running Experiment 2: Deeper Research...")
    results['exp2'] = await run_experiment(config_exp2, "Experiment 2: Deeper Research")
    
    # Run Experiment 3
    print("\n🔬 Running Experiment 3: Anthropic Native Search...")
    results['exp3'] = await run_experiment(config_exp3, "Experiment 3: Anthropic Native Search")
    
    # Run Experiment 4
    print("\n🔬 Running Experiment 4: Disabled Clarification...")
    results['exp4'] = await run_experiment(config_exp4, "Experiment 4: Disabled Clarification")
    
    # Display metrics summary
    display_metrics_table(results)
    
    # Run LLM comparative evaluation
    evaluation = await evaluate_all_reports(results, baseline_report)
    display_rankings(evaluation)
    
    # Save results to file for later reference
    print("\n" + "="*80)
    print("SAVING RESULTS")
    print("="*80)
    
    results_to_save = {
        exp_name: {
            "metrics": exp_data["metrics"],
            "report_preview": exp_data["final_report"][:500] + "..."
        }
        for exp_name, exp_data in results.items()
    }
    
    with open("experiment_results.json", "w") as f:
        json.dump(results_to_save, f, indent=2)
    
    print("✓ Results saved to experiment_results.json")
    print("\n" + "="*80)
    print("ALL EXPERIMENTS COMPLETE!")
    print("="*80)
    print("\nKey Takeaways:")
    print("1. Review the quantitative metrics to understand performance trade-offs")
    print("2. Check the LLM evaluation for qualitative insights")
    print("3. Consider which configuration best fits your use case")
    print("\nNext Steps:")
    print("- Copy sections to notebook for interactive exploration")
    print("- Try additional configuration combinations")
    print("- Analyze specific reports in detail")

In [ ]:
if __name__ == "__main__":
    asyncio.run(main())

# Run all experiments
results = {}

print("🔬 Running Experiment 1: Increased Parallelism...")
results['exp1'] = await run_experiment(config_exp1, "Experiment 1: Increased Parallelism")

print("🔬 Running Experiment 2: Deeper Research...")
results['exp2'] = await run_experiment(config_exp2, "Experiment 2: Deeper Research")

print("🔬 Running Experiment 3: Anthropic Native Search...")
results['exp3'] = await run_experiment(config_exp3, "Experiment 3: Anthropic Native Search")

print("🔬 Running Experiment 4: Disabled Clarification...")
results['exp4'] = await run_experiment(config_exp4, "Experiment 4: Disabled Clarification")

# Display metrics
display_metrics_table(results)

# Run LLM evaluation
evaluation = await evaluate_all_reports(results, baseline_report)
display_rankings(evaluation)